# Étape 1 – Extraction & Nettoyage

In [3]:
#imports et chargements
from pathlib import Path # la classe Path du module standard pathlib de Python : sert à manipuler les chemins de fichiers ou de dossiers de manière portable
import pandas as pd

raw = Path ("C:/Users/USER/Downloads/projet_personnel_fraude_bancaire/data/raw") # créer un objet path
df = pd.read_csv(raw/ "creditcard.csv")
df.shape , df.head()

((284807, 31),
    Time        V1        V2        V3        V4        V5        V6        V7  \
 0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
 1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
 2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
 3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
 4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   
 
          V8        V9  ...       V21       V22       V23       V24       V25  \
 0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
 1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
 2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
 3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
 4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   
 
   

### Étape 1.1 — Vérifier la structure et la qualité

In [5]:
df.info() #type de chaque colonnes et combien de valeurs maquantes
 
df.describe() # describe : donnes les statistiques de base

# Compter les valeurs manquantes par colonne
missing = df.isna().sum()

# Afficher seulement celles avec au moins 1 valeur manquante
print(missing[missing >0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

### Étape 1.2 — Supprimer les doublons
parfois le fichier contient la même transaction répétée, ce qui fausse l’analyse.

In [7]:
# Suppression des doublons (même ligne répétée)
avant = df.shape[0]    # 284807 lignes
df = df.drop_duplicates()
apres = df.shape[0]    # 283726 lignes

print(f"Doublons supprimés : {avant - apres}")   #1081 doublons
print(f"Taille du dataset maintenant : {df.shape}")

Doublons supprimés : 1081
Taille du dataset maintenant : (283726, 31)


### Étape 1.3 — Nettoyage des noms de colonnes

In [9]:
# Uniformiser les noms de colonnes (sans espace , minuscules)
df.columns = [col.strip().lower() for col in df.columns]
df.columns #renvoie la liste des noms de colonnes du DataFrame

Index(['time', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10',
       'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20',
       'v21', 'v22', 'v23', 'v24', 'v25', 'v26', 'v27', 'v28', 'amount',
       'class'],
      dtype='object')

### Étape 1.4 — Gérer les valeurs manquantes (NA)
* On supprime les colonnes très incomplètes.
* On remplace les valeurs manquantes dans les colonnes numériques pour ne pas perdre de lignes.

In [11]:
missing_pct = df.isna().mean() * 100   # vérifier le % des valeurs manquantes
print ("le pourcentage des valeurs manquantes par colonnes :")
print (missing_pct[missing_pct > 0].sort_values(ascending = False))  # garde seulement les colonnes où il y a au moins une valeur manquante et trie les colonnes du plus grand au plus petit pourcentage

# Si certaines colonnes ont >40% de valeurs manquantes, on peut les supprimer
cols_to_drop = missing_pct [missing_pct > 40].index.tolist()
df = df.drop(columns = cols_to_drop)

# Remplacer les NA restants par la médiane (pour les colonnes numériques)
for col in df.select_dtypes(exclude = "object").columns :   #sélectionne uniquement les colonnes numériques et récupère la liste des noms de ces colonnes
    if df[col].isna().any():                                #renvoie True si au moins une valeur est manquante dans la colonne
        df[col] = df[col].fillna(df[col].median())          #remplace les NaN la médiane de la colonne

le pourcentage des valeurs manquantes par colonnes :
Series([], dtype: float64)


### Étape 1.5 — Création de nouvelles colonnes utiles
* **heure** et **jour_simule →** permettent de repérer à quel moment les fraudes se produisent
* **amount_log1p →** rend la distribution des montants plus régulière (utile pour graphiques)
* **is_fraud →** plus clair que class (0 ou 1)

In [13]:
import numpy as np

# Créer la colonne "heure" à partir du temps (en secondes)
if "time" in df.columns :
    df["heure"] = (df["time"] // 3600).astype(int) %24  #heure de 0 à 24
    df["jour_simule"] =( df["time"] // (3600*24)).astype(int)  #jour simulé
    
# Transformation du montant
if "amount" in df.columns :
    df["amount_log1p"] =np.log1p(df["amount"])                # transformation logarithmique

# Colonne binaire pour la fraude
if "class" in df.columns :
    df["is_fraud"] = df["class"].astype(int)

### Étape 1.6 — Vérification finale du nettoyage

In [21]:
print ("Taille finale :",df.shape)
print ("Nombre de NA restant :",df.isna().sum())
print ("Taux de Fraude :",df["is_fraud"].mean())
df.head()

Taille finale : (283726, 35)
Nombre de NA restant : time            0
v1              0
v2              0
v3              0
v4              0
v5              0
v6              0
v7              0
v8              0
v9              0
v10             0
v11             0
v12             0
v13             0
v14             0
v15             0
v16             0
v17             0
v18             0
v19             0
v20             0
v21             0
v22             0
v23             0
v24             0
v25             0
v26             0
v27             0
v28             0
amount          0
class           0
heure           0
jour_simule     0
amount_log1p    0
is_fraud        0
dtype: int64
Taux de Fraude : 0.001667101358352777


,time,v1,v2,v3,v4,v5,v6,v7,v8,v9,...,v25,v26,v27,v28,amount,class,heure,jour_simule,amount_log1p,is_fraud
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0,0,0,5.014760,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,0,0,0,1.305626,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,0,0,5.939276,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0,0,0,4.824306,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,0,0,0,4.262539,0


### Étape 1.7 — Sauvegarde du fichier propre

In [24]:
from pathlib import Path

processed = Path("C:/Users/USER/Downloads/projet_personnel_fraude_bancaire/data/processed")
processed.mkdir(parents=True, exist_ok=True)

output_file = processed / "fraude_clean.csv"
df.to_csv(output_file, index=False)

print ("fichier nettoyé et sauvegardé : {output_file}")

fichier nettoyé et sauvegardé : {output_file}
